# Sister Forecast + QR



## 1. Setup

In [3]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import statsmodels.api as sm

REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "requirements").exists():
    REPO_ROOT = REPO_ROOT.parent
ROOT = REPO_ROOT / "load"

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from src.metrics import print_metrics

DATA_DIR = ROOT / "data"
OUTPUT_DIR = ROOT / "outputs" / "predictions"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = DATA_DIR / "Load-Train.xlsx"
TEST_PATH = DATA_DIR / "Load-Test.xlsx"
OUTPUT_PATH = OUTPUT_DIR / "sister_qra.csv"

MAIN_QUANTILES = [0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9]
TAIL_QUANTILES = [0.01,0.025,0.05,0.95,0.975,0.99]
ALL_QUANTILES = MAIN_QUANTILES + TAIL_QUANTILES
TEST_START = pd.Timestamp("2022-01-01")

## 2. Load and preprocess

In [4]:
def prepare_raw_load(df, base_year=2019):
    out = df.copy()
    out["timestamp"] = pd.to_datetime(
        {
            "year": base_year + out["Year"],
            "month": out["Month"],
            "day": out["Day"],
            "hour": out["Hour"] - 1,
        },
        errors="raise",
    )
    out = out.rename(columns={"Load": "target"})
    return out

context_df = prepare_raw_load(pd.read_excel(TRAIN_PATH))
test_df = prepare_raw_load(pd.read_excel(TEST_PATH))
df = pd.concat([context_df, test_df], ignore_index=True).sort_values("timestamp").reset_index(drop=True)

df["load"] = df["target"]
df["temp"] = df[[
    "Site-1 Temp","Site-2 Temp","Site-3 Temp","Site-4 Temp","Site-5 Temp"
]].mean(axis=1)
df["ghi"] = df[[
    "Site-1 GHI","Site-2 GHI","Site-3 GHI","Site-4 GHI","Site-5 GHI"
]].mean(axis=1)

print("Train rows:", len(context_df))
print("Test rows:", len(test_df))
df.head()

Train rows: 17544
Test rows: 8760


,Year,Month,Day,Hour,target,Site-1 Temp,Site-2 Temp,Site-3 Temp,Site-4 Temp,Site-5 Temp,Site-1 GHI,Site-2 GHI,Site-3 GHI,Site-4 GHI,Site-5 GHI,timestamp,load,temp,ghi
0,1,1,1,1,1997,8.0,8.2,5.3,9.4,8.1,0,0,0,0,0,2020-01-01 00:00:00,1997,7.80,0.0
1,1,1,1,2,1921,8.3,8.6,5.2,8.6,7.1,0,0,0,0,0,2020-01-01 01:00:00,1921,7.56,0.0
2,1,1,1,3,1861,8.1,8.8,5.1,8.7,6.2,0,0,0,0,0,2020-01-01 02:00:00,1861,7.38,0.0
3,1,1,1,4,1833,7.6,8.1,4.3,8.5,6.0,0,0,0,0,0,2020-01-01 03:00:00,1833,6.90,0.0
4,1,1,1,5,1847,7.3,7.5,4.0,8.6,6.9,0,0,0,0,0,2020-01-01 04:00:00,1847,6.86,0.0


## 3. Model

In [5]:
def build_base_formula():
    return (
        "load ~ C(month) + C(dow) + C(hour) + C(dow):C(hour)"
        " + temp + I(temp**2) + I(temp**3)"
        " + C(month):temp + C(month):I(temp**2) + C(month):I(temp**3)"
        " + C(hour):temp + C(hour):I(temp**2) + C(hour):I(temp**3)"
        " + ghi + I(ghi**2) + C(hour):ghi"
    )


def add_temp_lags(df, lags):
    for lag in lags:
        df[f"temp_lag_{lag}"] = df["temp"].shift(lag)
    return df


def add_temp_daily_avg(df, max_d):
    for d in range(1, max_d + 1):
        start = 24*d - 23
        df[f"temp_avg_{d}"] = (
            df["temp"]
            .shift(start)
            .rolling(24)
            .mean()
        )
    return df


def build_sister_formula(base_formula, lags, avg_days):
    terms = []
    for lag in lags:
        terms += [
            f"temp_lag_{lag}",
            f"I(temp_lag_{lag}**2)",
            f"I(temp_lag_{lag}**3)",
        ]
    for d in avg_days:
        terms += [
            f"temp_avg_{d}",
            f"I(temp_avg_{d}**2)",
            f"I(temp_avg_{d}**3)",
        ]
    if terms:
        return base_formula + " + " + " + ".join(terms)
    return base_formula

In [6]:
def train_sister_models(df):
    sister_configs = [
        {"name":"m1","lags":[1,2,3],"avg_days":[1]},
        {"name":"m2","lags":[1,2,3,6],"avg_days":[1]},
        {"name":"m3","lags":[1,2,3,6,12],"avg_days":[1]},
        {"name":"m4","lags":[1,2,3,6,12,24],"avg_days":[1]},
        {"name":"m5","lags":[1,2,3],"avg_days":[1,2]},
        {"name":"m6","lags":[1,2,3,6],"avg_days":[1,2]},
    ]

    base_formula = build_base_formula()
    models = {}
    for cfg in sister_configs:
        formula = build_sister_formula(base_formula, cfg["lags"], cfg["avg_days"])
        model = smf.ols(formula=formula, data=df, missing="drop").fit()
        models[cfg["name"]] = model
    return models, sister_configs


def generate_sister_predictions(df, models):
    df = df.copy()
    for name, model in models.items():
        df[name] = model.predict(df)
    return df


def fit_qra(df, sister_cols, quantiles):
    X = sm.add_constant(df[sister_cols])
    y = df["load"]
    models = {}
    for q in quantiles:
        mod = sm.QuantReg(y, X)
        models[q] = mod.fit(q=q)
    return models


def predict_qra(df, qra_models, sister_cols):
    X = sm.add_constant(df[sister_cols])
    for q, model in qra_models.items():
        df[f"q_{q}"] = model.predict(X)
    return df

## 4. Train 

In [7]:
def qra_pipeline(df):
    work = df.copy()
    work["timestamp"] = pd.to_datetime(work["timestamp"])
    work = work.sort_values("timestamp")

    work["hour"] = work.timestamp.dt.hour
    work["dow"] = work.timestamp.dt.dayofweek
    work["month"] = work.timestamp.dt.month

    # Exact source feature construction.
    work = add_temp_lags(work, [1,2,3,6,12,24])
    work = add_temp_daily_avg(work, max_d=2)
    work = work.dropna()

    train = work[work.timestamp < TEST_START]
    models, configs = train_sister_models(train)
    sister_cols = [c["name"] for c in configs]

    train_pred = generate_sister_predictions(train, models)
    qra_models = fit_qra(train_pred, sister_cols, ALL_QUANTILES)

    test = work[work.timestamp >= TEST_START]
    preds = []
    for day in pd.date_range("2022-01-01", "2022-12-31", freq="D"):
        day_df = test[(test.timestamp >= day) & (test.timestamp < day + pd.Timedelta(days=1))].copy()
        if len(day_df) == 0:
            continue
        day_df = generate_sister_predictions(day_df, models)
        day_df = predict_qra(day_df, qra_models, sister_cols)
        preds.append(day_df)

    return pd.concat(preds, ignore_index=True)

raw_forecasts = qra_pipeline(df)
raw_forecasts.head()

/tmp/ipykernel_5223/4291719162.py:15: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  model = smf.ols(formula=formula, data=df, missing="drop").fit()
/tmp/ipykernel_5223/4291719162.py:15: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  model = smf.ols(formula=formula, data=df, missing="drop").fit()
/tmp/ipykernel_5223/4291719162.py:15: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  model = smf.ols(formula=formula, data=df, missing="drop").fit()
/tmp/ipykernel_5223/4291719162.py:15: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  model = smf.ols(formula=formula, data=df, missing="drop").fit()
/tmp/ipykernel_5223/4291719162.py:15: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determin

,Year,Month,Day,Hour,target,Site-1 Temp,Site-2 Temp,Site-3 Temp,Site-4 Temp,Site-5 Temp,...,q_0.6,q_0.7,q_0.8,q_0.9,q_0.01,q_0.025,q_0.05,q_0.95,q_0.975,q_0.99
0,3,1,1,1,2064,12.3,11.5,9.3,12.7,10.8,...,1930.073340,1951.969169,1977.931551,2015.309926,1607.461150,1672.855595,1729.968171,2049.965494,2080.952644,2130.856527
1,3,1,1,2,1975,12.1,11.1,9.3,12.3,10.6,...,1868.547192,1890.016624,1915.604960,1954.220583,1560.893704,1625.722893,1679.103525,1989.947162,2022.990154,2078.007327
2,3,1,1,3,1900,10.8,10.2,9.0,11.8,9.9,...,1833.752011,1854.929031,1880.396061,1919.428855,1532.299165,1595.981885,1648.137903,1955.353037,1989.451075,2046.293464
3,3,1,1,4,1829,10.0,10.3,9.3,10.9,8.9,...,1811.491339,1832.222727,1857.947720,1898.018025,1514.153067,1572.713968,1624.177976,1934.379591,1971.714950,2032.403171
4,3,1,1,5,1828,11.1,10.3,9.8,10.9,9.6,...,1833.845354,1854.886760,1882.376911,1927.295951,1547.789149,1604.153225,1650.925471,1969.893181,2017.080290,2092.325678


## 5. Eval

In [8]:
def standardize_sister_output(forecasts):
    out = forecasts.copy()
    for q in ALL_QUANTILES:
        out[str(q)] = out[f"q_{q}"]
    out["prediction"] = out["0.5"]

    keep = ["timestamp", "target", "prediction"] + [str(q) for q in ALL_QUANTILES]
    return out[keep].sort_values("timestamp").reset_index(drop=True)

result_df = standardize_sister_output(raw_forecasts)
metrics = print_metrics(result_df, quantiles=MAIN_QUANTILES, label="Sister Forecast + QRA")
result_df.head()

Sister Forecast + QRA
MAE:  108.547888
RMSE: 143.935398
aQL:  43.511533


,timestamp,target,prediction,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,0.01,0.025,0.05,0.95,0.975,0.99
0,2022-01-01 00:00:00,2064,1908.447118,1783.119011,1833.280943,1862.740012,1886.793362,1908.447118,1930.073340,1951.969169,1977.931551,2015.309926,1607.461150,1672.855595,1729.968171,2049.965494,2080.952644,2130.856527
1,2022-01-01 01:00:00,1975,1847.484151,1728.122162,1775.349756,1803.511291,1826.742500,1847.484151,1868.547192,1890.016624,1915.604960,1954.220583,1560.893704,1625.722893,1679.103525,1989.947162,2022.990154,2078.007327
2,2022-01-01 02:00:00,1900,1812.961243,1695.582659,1742.083637,1769.867471,1792.684696,1812.961243,1833.752011,1854.929031,1880.396061,1919.428855,1532.299165,1595.981885,1648.137903,1955.353037,1989.451075,2046.293464
3,2022-01-01 03:00:00,1829,1790.709087,1671.347195,1719.470024,1747.967994,1771.038479,1790.709087,1811.491339,1832.222727,1857.947720,1898.018025,1514.153067,1572.713968,1624.177976,1934.379591,1971.714950,2032.403171
4,2022-01-01 04:00:00,1828,1813.262807,1693.373954,1741.806717,1771.221720,1794.662464,1813.262807,1833.845354,1854.886760,1882.376911,1927.295951,1547.789149,1604.153225,1650.925471,1969.893181,2017.080290,2092.325678


In [9]:
result_df.to_csv(OUTPUT_PATH, index=False)


Saved: /content/drive/MyDrive/energy_tsfm_benchmark/load/outputs/predictions/sister_qra.csv
